<a href="https://colab.research.google.com/github/bogdanbabych/experiments_NLTK/blob/main/Sketch_Engine_Cockpit_VA_v01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sketch-Engine-Wrapper - Colab-Cockpit

**Aufbau:** Dieses Notebook ist das *Cockpit*. Die *Engine* (Client, Config,
SubcorpusManager, Aufgaben **c**, **d**, **e**, **f**, Kanal **Z**) wird von Zelle 2 per
`%%writefile` als `sketch_engine_wrapper.py` geschrieben und importiert.

**Verifiziert:** corp_info-Groesse, gramrels-Namen, `ws_status='ok'`=fertig,
wsketch `word`/`count`/`score`, `Constructions`-Filter, wordlist `Items`/`str`/`frq`,
collx `Stats` (namensbasiert d/m/t) + `freq`/`coll_freq`, Komposita ohne `(?i)`.
**Neu zu bestaetigen:** extract_keywords-Struktur + simple_n-Spielraum + subcorp
(Aufgabe e, unten).

**Vorgehen:** Zellen **1-3**, dann **Smoke-Test** (4-9), dann **produktiver Lauf**
(10-11). Der API-Key wird per `getpass`/Colab-Secret bezogen und **nirgends
ausgegeben**.

> **Engine aktualisieren:** Inhalt von Zelle 2 ersetzen, Zellen 2-3 neu ausfuehren.


In [ ]:
# (1) Abhaengigkeiten
!pip install -q pandas openpyxl requests

In [ ]:
%%writefile sketch_engine_wrapper.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Generischer Sketch-Engine-Wrapper (Colab + importierbares Modul)
================================================================

Wiederverwendbares Extraktionswerkzeug fuer die Sketch-Engine-REST-API.
Projekt- und hypothesenunabhaengig: kennt keine Zielwoerter, keine Achsen,
keine Forschungsfrage. Es zieht Daten und gibt sie strukturiert aus.

Dieser Build liefert das FUNDAMENT plus zwei Aufgaben:
    * SEClient          - Auth, Throttling, Async-Polling, Fehler-/Leerfall
    * SubcorpusManager  - Query-Subkorpora anlegen/pollen/auflisten/loeschen
                          (Grundlage fuer e/f; Subkorpora bleiben erhalten)
    * Config            - interaktiv, bedarfsgesteuert, als JSON gespeichert
    * Aufgabe c         - Word Sketch        (Endpunkt wsketch)
    * Aufgabe d         - Fensterkollokation (Endpunkt collx)
    * Aufgabe e         - Schluesselwoerter  (extract_keywords) + Diskursmodus
    * Aufgabe f         - Mehrwort-Schluesseleinheiten (extract_keywords+usengrams)
    * Kanal Z           - Komposita/Wortbildung (Endpunkt wordlist)
    * Output            - XLSX-Workbooks (+ CSVs), Master und/oder pro Lexem

Alle Aufgaben c-f und Z sind implementiert und gegen die Live-API verifiziert.

------------------------------------------------------------------------
VERIFIZIERT (Smoke-Test, vgl. Spezifikation Sec. 2)
------------------------------------------------------------------------
* Base https://api.sketchengine.eu/search ; Auth Basic (USERNAME, API_KEY)
* wsketch -> Top-Level 'Gramrels' [ ].'Words' [ ] mit word/count/score
  (count = Kookkurrenzfrequenz, score = logDice). Erste Relation
  'Constructions' ist Verteilungsstatistik -> herausfiltern. '%w' im
  Relationsnamen = Platzhalter fuers Zielwort.
* corp_info?gramrels=1 -> Relationsnamen als Strings.
* concordance/wsketch: grosse Korpora liefern asynchron -> pollen.
* Fair Use: serielle Calls mit Pause (Throttling).

ZU VERIFIZIEREN IM ERSTEN LIVE-LAUF (defensiv geparst, leicht anpassbar;
mit "VERIFY" markiert): exakter corp_info-Groessenschluessel; wsketch-
Vollstaendigkeitssignal (ws_status); wordlist-Item-Schluessel; subcorp-
Anlage-Parameternamen (create/q/struct) und subcorp_info-Groesse;
gramrels=1-Antwortschluessel.

------------------------------------------------------------------------
SICHERHEIT
------------------------------------------------------------------------
Der API-Key wird ueber getpass / Colab-Secrets / Umgebungsvariable bezogen
und NIRGENDS ausgegeben (nicht in Logs, nicht im Config-Output, nicht im
gespeicherten JSON). Das Skript laeuft beim Nutzer (lokal/Colab).
"""

from __future__ import annotations

import os
import re
import sys
import json
import time
import threading
import getpass
import datetime as _dt
from typing import Any, Dict, List, Optional, Tuple

import requests

try:
    import pandas as pd
except ImportError:  # pragma: no cover
    pd = None  # wird in _ensure_deps geprueft


# ============================================================================
# Konstanten / Endpunkte / Defaults
# ============================================================================

BASE_URL = "https://api.sketchengine.eu/search"

ENDPOINTS = {
    "corp_info": "corp_info",
    "wsketch": "wsketch",
    "wordlist": "wordlist",
    "concordance": "concordance",
    "collx": "collx",
    "extract_keywords": "extract_keywords",
    "subcorp": "subcorp",
    "subcorp_info": "subcorp_info",
}

# Aufgaben-Register: Kuerzel -> (Klartext, Eingabe-Klasse)
#   node      = knotenzentriert (braucht Zielwortliste)
#   contrast  = korpuskontrastiv (braucht Referenzkorpus)
TASKS = {
    "c": ("Word Sketch", "node"),
    "d": ("Fensterkollokation", "node"),
    "e": ("Schluesselwoerter (Einwort)", "contrast"),
    "f": ("Mehrwort-Schluesseleinheiten", "contrast"),
    "Z": ("Komposita / Wortbildung", "node"),
}

# Klarnamen fuer Ausgabedateien (CSV) und XLSX-Sheets (Sheet <= 31 Zeichen).
TASK_FILENAME = {
    "c": "c_word_sketch",
    "d": "d_window_collocation",
    "e": "e_keywords",
    "f": "f_multiword_keywords",
    "Z": "Z_compounds",
}
TASK_SHEET = {
    "c": "c_word_sketch",
    "d": "d_collocation",
    "e": "e_keywords",
    "f": "f_multiword_keys",
    "Z": "Z_compounds",
}
# Kurze Art-Kuerzel fuer Dateinamen (Konvention: Lemma__Art__Korpus_Zeitstempel).
TASK_TAG = {
    "c": "wsketch",
    "d": "coll",
    "e": "kw",
    "f": "mwkw",
    "Z": "komposita",
}

# Default-Parameter (vgl. Spezifikation Sec. 5)
DEFAULTS = {
    "throttle_seconds": 1.0,      # Pause zwischen Calls (Fair Use)
    "poll_interval": 3.0,         # Sekunden zwischen Async-Polls
    "poll_timeout": 600.0,        # harter Timeout in Sekunden -> Fehler statt Teildaten
    "wsketch_maxitems": 50,       # maxitems je Relation
    "clustercolls": 0,
    "coll_window": (-5, 5),       # Fenster fuer d
    "cbgrfns": ["d", "m", "t"],   # logDice + MI + T-Score
    "n_collocates": 50,
    "wl_maxitems": 1000,          # Kandidatenpool fuer Komposita (Z)
    "wlicase": 0,                 # 0 = case-insensitiv (Komposita-Suche)
    # Frisch angelegte (grosse) Subkorpora sind nicht sofort abfragebereit:
    # extract_keywords gibt 0 zurueck, bis SE das Subkorpus kompiliert hat.
    # Bei subcsize>0 -> warten und erneut versuchen (Zeitbudget, wachsendes Intervall).
    "kw_ready_timeout": 120.0,    # max. Wartezeit je Subkorpus auf Abfragebereitschaft (dann weiter)
    "kw_ready_wait": 6.0,         # Start-Wartezeit zwischen Versuchen (waechst x1.5, max 30s)
    # Diskurs-Subkorpus, das einen zu grossen Anteil am Korpus ausmacht, macht die
    # Keyness flach (Referenz enthaelt den Fokus). Ueber dieser Schwelle: nachfragen.
    "kw_max_subcorpus_share": 0.05,        # 5 % des Korpus
    "kw_large_subcorpus_action": "ask",    # ask | skip | force
    "kw_ask_timeout": 60.0,                # Sekunden bis Default-Nein bei 'ask'
}

# Umgebungsvariablen / Colab-Secret-Namen fuer die Auth
ENV_USER = "SKETCHENGINE_USERNAME"
ENV_KEY = "SKETCHENGINE_API_KEY"
COLAB_USER = "SKETCH_ENGINE_USERNAME"
COLAB_KEY = "SKETCH_ENGINE_API_KEY"


# ============================================================================
# Hilfsfunktionen: Abhaengigkeiten, Zeit, sichere Parser
# ============================================================================

def _ensure_deps() -> None:
    """Stellt sicher, dass pandas + openpyxl vorhanden sind (Colab: meist da)."""
    global pd
    if pd is None:
        raise ImportError(
            "pandas wird benoetigt. In Colab/lokal:  pip install pandas openpyxl"
        )
    try:
        import openpyxl  # noqa: F401
    except ImportError as exc:  # pragma: no cover
        raise ImportError(
            "openpyxl wird fuer den XLSX-Export benoetigt:  pip install openpyxl"
        ) from exc


def _now_stamp() -> str:
    return _dt.datetime.now().strftime("%Y%m%d_%H%M%S")


def _now_iso() -> str:
    return _dt.datetime.now().isoformat(timespec="seconds")


def _sanitize(name: str) -> str:
    """Macht ein Lemma fuer Subkorpus-/Dateinamen tauglich."""
    return re.sub(r"[^0-9A-Za-z_]+", "_", name).strip("_") or "x"


def _corpus_tag(corpname: str) -> str:
    """
    Kurzes, identifizierendes Kuerzel fuers Korpus, fuer Dateinamen.
    'user/LS_Fr_It_IUED/all_v_d_k' -> 'LS_Fr_It_IUED_all_v_d_k'
    'preloaded/detenten23_rft3_1'  -> 'detenten23_rft3_1'
    """
    parts = [p for p in str(corpname).split("/") if p]
    if parts and parts[0] in ("user", "preloaded"):
        parts = parts[1:] or [str(corpname)]
    return _sanitize("_".join(parts))


_UMLAUT = {"ä": "ae", "ö": "oe", "ü": "ue", "Ä": "Ae", "Ö": "Oe",
           "Ü": "Ue", "ß": "ss"}


def _file_part(s: str) -> str:
    """Lexem fuer Dateinamen: Umlaute transliteriert, Rest konservativ bereinigt."""
    for k, v in _UMLAUT.items():
        s = s.replace(k, v)
    return re.sub(r"[^0-9A-Za-z_-]+", "_", s).strip("_") or "x"


def _cfg(cfg: Dict[str, Any], *keys, default=None):
    """Erster vorhandener Config-Schluessel (neue Namen mit Alt-Namen-Fallback)."""
    for k in keys:
        if k in cfg and cfg[k] is not None:
            return cfg[k]
    return default


def _first_present(d: Dict[str, Any], keys: List[str], default=None):
    """Erster vorhandener Schluessel aus einer Kandidatenliste (defensives Parsen)."""
    for k in keys:
        if isinstance(d, dict) and k in d and d[k] not in (None, ""):
            return d[k]
    return default


def _to_int(value: Any, default: Optional[int] = None) -> Optional[int]:
    try:
        return int(float(str(value).replace(",", "").replace(" ", "")))
    except (TypeError, ValueError):
        return default


def _to_float(value: Any, default: Optional[float] = None) -> Optional[float]:
    try:
        return float(str(value).replace(",", "."))
    except (TypeError, ValueError):
        return default


# ============================================================================
# Groessenbewusste Default-Vorschlaege fuer Frequenzschwellen
# ============================================================================

def size_tier_defaults(token_count: Optional[int]) -> Dict[str, int]:
    """
    Liefert groessenabhaengige Vorschlaege fuer die Frequenzschwellen.
    Eine feste 5 ist fuer ein kleines Fachkorpus richtig, fuer ein
    Milliarden-Token-Korpus viel zu niedrig (vgl. Spezifikation Sec. 2/3).
    Alle Werte sind Vorschlaege und ueberschreibbar.
    """
    if token_count is None:
        # unbekannt -> konservativ klein
        return {"cminfreq": 5, "cminbgr": 5, "kw_minfreq": 5, "wlminfreq": 5}
    if token_count < 10_000_000:            # < 10 Mio.
        return {"cminfreq": 5, "cminbgr": 5, "kw_minfreq": 5, "wlminfreq": 3}
    if token_count < 1_000_000_000:         # 10 Mio. - 1 Mrd.
        return {"cminfreq": 15, "cminbgr": 10, "kw_minfreq": 20, "wlminfreq": 10}
    return {"cminfreq": 50, "cminbgr": 25, "kw_minfreq": 100, "wlminfreq": 50}


def _format_tokens(n: Optional[int]) -> str:
    if n is None:
        return "unbekannt"
    if n >= 1_000_000_000:
        return f"{n/1_000_000_000:.2f} Mrd. Token"
    if n >= 1_000_000:
        return f"{n/1_000_000:.2f} Mio. Token"
    return f"{n} Token"


# ============================================================================
# Auth-Aufloesung (Colab-Secret -> Umgebungsvariable -> getpass)
# ============================================================================

def resolve_credentials(
    username: Optional[str] = None,
    api_key: Optional[str] = None,
) -> Tuple[str, str]:
    """
    Loest (USERNAME, API_KEY) auf. Reihenfolge:
        1. explizit uebergebene Argumente
        2. Colab-Secrets (google.colab.userdata)
        3. Umgebungsvariablen
        4. interaktive Eingabe (getpass fuer den Key)
    Der Key wird NIE ausgegeben.
    """
    # 2. Colab-Secrets
    if username is None or api_key is None:
        try:
            from google.colab import userdata  # type: ignore
            if username is None:
                try:
                    username = userdata.get(COLAB_USER)
                except Exception:
                    username = None
            if api_key is None:
                try:
                    api_key = userdata.get(COLAB_KEY)
                except Exception:
                    api_key = None
        except Exception:
            pass

    # 3. Umgebungsvariablen
    if username is None:
        username = os.environ.get(ENV_USER)
    if api_key is None:
        api_key = os.environ.get(ENV_KEY)

    # 4. interaktiv
    if not username:
        username = input("Sketch-Engine-Benutzername: ").strip()
    if not api_key:
        api_key = getpass.getpass("Sketch-Engine-API-Key (verdeckt): ").strip()

    if not username or not api_key:
        raise ValueError("Benutzername und API-Key sind erforderlich.")
    return username, api_key


# ============================================================================
# Fehlertypen
# ============================================================================

class SketchEngineError(RuntimeError):
    """Allgemeiner API-Fehler (HTTP- oder SE-Fehlerobjekt)."""


class SketchEngineTimeout(SketchEngineError):
    """Async-Berechnung ueberschreitet den harten Timeout."""


# ============================================================================
# SE-Client
# ============================================================================

class SEClient:
    """
    Schlanke Client-Klasse: Auth, Throttling, Async-Polling, Fehler-/Leerfall.
    Jeder Aufruf ist ein serieller GET mit Basic-Auth und Pause danach.
    """

    def __init__(
        self,
        username: str,
        api_key: str,
        throttle_seconds: float = DEFAULTS["throttle_seconds"],
        poll_interval: float = DEFAULTS["poll_interval"],
        poll_timeout: float = DEFAULTS["poll_timeout"],
        verbose: bool = True,
    ):
        self._auth = requests.auth.HTTPBasicAuth(username, api_key)
        self.username = username  # der Key wird bewusst NICHT als Attribut gehalten
        self.throttle_seconds = throttle_seconds
        self.poll_interval = poll_interval
        self.poll_timeout = poll_timeout
        self.verbose = verbose
        self._session = requests.Session()
        self._last_call = 0.0

    # ----- Low-Level -------------------------------------------------------

    def _log(self, msg: str) -> None:
        if self.verbose:
            print(msg, flush=True)

    def _throttle(self) -> None:
        elapsed = time.time() - self._last_call
        wait = self.throttle_seconds - elapsed
        if wait > 0:
            time.sleep(wait)

    def _raw_get(self, endpoint: str, params: Dict[str, Any]) -> Dict[str, Any]:
        """Ein einzelner GET. Wirft bei HTTP- oder SE-Fehlern."""
        url = f"{BASE_URL}/{ENDPOINTS.get(endpoint, endpoint)}"
        # 'format' immer auf json, sofern nicht explizit gesetzt
        params = {k: v for k, v in params.items() if v is not None}
        params.setdefault("format", "json")
        self._throttle()
        try:
            resp = self._session.get(url, params=params, auth=self._auth, timeout=300)
        except requests.RequestException as exc:
            raise SketchEngineError(f"Netzwerkfehler bei '{endpoint}': {exc}") from exc
        finally:
            self._last_call = time.time()

        if resp.status_code == 401:
            raise SketchEngineError(
                "401 Unauthorized - Benutzername/API-Key pruefen (Auth fehlgeschlagen)."
            )
        if resp.status_code >= 400:
            raise SketchEngineError(
                f"HTTP {resp.status_code} bei '{endpoint}': {resp.text[:300]}"
            )
        try:
            data = resp.json()
        except ValueError as exc:
            raise SketchEngineError(
                f"Antwort bei '{endpoint}' ist kein JSON: {resp.text[:300]}"
            ) from exc

        # SE meldet Fehler oft als Feld 'error' im 200er-Body
        if isinstance(data, dict) and data.get("error"):
            raise SketchEngineError(f"SE-Fehler bei '{endpoint}': {data['error']}")
        return data

    # ----- Async-Polling ---------------------------------------------------

    # Status-Strings, die "Berechnung laeuft noch" bedeuten. Numerischer
    # Fortschritt (< 1.0) wird separat als laufend erkannt. VERIFIED im
    # Smoke-Test: ws_status == 'ok' bedeutet FERTIG. Der exakte Running-
    # String fuer sehr grosse Korpora ist der einzige Restunbekannte und
    # bei Bedarf hier zu ergaenzen.
    _RUNNING_STATUS = {"running", "computing", "processing", "wait",
                       "waiting", "scheduled", "pending"}

    @classmethod
    def _wsketch_complete(cls, data: Dict[str, Any]) -> bool:
        """
        True, wenn die Berechnung fertig ist. 'ws_status' ist die Autoritaet:
        'ok'/'finished'/fehlend -> fertig; numerischer Wert < 1.0 oder ein
        bekannter Running-String -> laeuft noch. Ein leeres Ergebnis (z.B.
        Wort ohne Kollokate) ist bei fertigem Status ein gueltiges Ergebnis.
        """
        if not isinstance(data, dict):
            return True
        status = data.get("ws_status")
        if status is None:
            return True
        try:
            return float(status) >= 1.0   # numerischer Fortschritt
        except (TypeError, ValueError):
            pass
        return str(status).strip().lower() not in cls._RUNNING_STATUS

    def _get_with_poll(
        self, endpoint: str, params: Dict[str, Any], payload_key: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        GET mit Async-Polling. Pollt, solange 'ws_status' "laeuft noch"
        signalisiert, sonst Rueckgabe (auch bei leerer Nutzlast - das ist ein
        gueltiges Ergebnis). Harter Timeout -> Fehler statt stiller Teildaten.
        """
        deadline = time.time() + self.poll_timeout
        attempt = 0
        while True:
            attempt += 1
            data = self._raw_get(endpoint, params)
            if self._wsketch_complete(data):
                if attempt > 1:
                    self._log(f"  [poll] '{endpoint}' nach {attempt} Versuchen fertig.")
                return data
            if time.time() >= deadline:
                raise SketchEngineTimeout(
                    f"'{endpoint}' nicht fertig nach {self.poll_timeout:.0f}s "
                    f"(ws_status={data.get('ws_status')!r}). Abbruch statt Teildaten."
                )
            self._log(
                f"  [poll] '{endpoint}' laeuft noch "
                f"(ws_status={data.get('ws_status')!r}); warte {self.poll_interval:.0f}s ..."
            )
            time.sleep(self.poll_interval)

    # ----- Endpunkt-Methoden ----------------------------------------------

    def corp_info(
        self,
        corpname: str,
        gramrels: int = 0,
        subcorpora: int = 0,
    ) -> Dict[str, Any]:
        return self._raw_get(
            "corp_info",
            {"corpname": corpname, "gramrels": gramrels, "subcorpora": subcorpora},
        )

    def corpus_size(self, corpname: str) -> Optional[int]:
        """
        Tokenzahl des Korpus. VERIFY: exakter Schluessel im Live-Lauf.
        Probiert mehrere bekannte Stellen, degradiert zu None.
        """
        info = self.corp_info(corpname)
        sizes = info.get("sizes", {}) if isinstance(info, dict) else {}
        candidate = _first_present(
            sizes, ["tokencount", "normsum", "wordcount"]
        )
        if candidate is None:
            candidate = _first_present(info, ["wordcount", "size", "tokencount"])
        return _to_int(candidate)

    def gramrel_names(self, corpname: str) -> List[str]:
        """
        Definierte Relationsnamen der Sketch-Grammar (Strings).
        VERIFY: Antwortschluessel ('Gramrels' vs. 'gramrels').
        """
        info = self.corp_info(corpname, gramrels=1)
        raw = _first_present(info, ["Gramrels", "gramrels", "gramrel_names"], [])
        names: List[str] = []
        for item in raw or []:
            if isinstance(item, str):
                names.append(item)
            elif isinstance(item, dict):
                nm = _first_present(item, ["name", "n"])
                if nm:
                    names.append(nm)
        return names

    def list_subcorpora(self, corpname: str) -> List[Dict[str, Any]]:
        """Benannte Subkorpora des Korpus (Name + ggf. Groesse)."""
        info = self.corp_info(corpname, subcorpora=1)
        raw = _first_present(info, ["subcorpora", "Subcorplist", "subcorplist"], [])
        out: List[Dict[str, Any]] = []
        for item in raw or []:
            if isinstance(item, dict):
                name = _first_present(item, ["n", "name", "subcname"])
                size = _to_int(_first_present(item, ["size", "tokencount", "cnt"]))
                if name:
                    out.append({"name": name, "size": size})
            elif isinstance(item, str):
                out.append({"name": item, "size": None})
        return out

    def wsketch(
        self,
        corpname: str,
        lemma: str,
        lpos: Optional[str] = None,
        usesubcorp: Optional[str] = None,
        maxitems: int = DEFAULTS["wsketch_maxitems"],
        clustercolls: int = DEFAULTS["clustercolls"],
        minfreq: Optional[str] = None,
    ) -> Dict[str, Any]:
        params = {
            "corpname": corpname,
            "lemma": lemma,
            "lpos": lpos or None,
            "usesubcorp": usesubcorp or None,
            "maxitems": maxitems,
            "clustercolls": clustercolls,
            "minfreq": minfreq,  # None -> SE-Default; 'auto' moeglich
        }
        return self._get_with_poll("wsketch", params, payload_key="Gramrels")

    def wordlist(
        self,
        corpname: str,
        wlattr: str = "lemma",
        wlpat: str = ".*",
        wlminfreq: int = 5,
        wlmaxitems: int = DEFAULTS["wl_maxitems"],
        wlicase: int = DEFAULTS["wlicase"],
        wlnums: str = "frq",
        usesubcorp: Optional[str] = None,
        include_nonwords: int = 0,
    ) -> Dict[str, Any]:
        params = {
            "corpname": corpname,
            "wlattr": wlattr,
            "wlpat": wlpat,
            "wlminfreq": wlminfreq,
            "wlmaxitems": wlmaxitems,
            "wlicase": wlicase,
            "wlnums": wlnums,
            "wlsort": wlnums,
            "usesubcorp": usesubcorp or None,
            "include_nonwords": include_nonwords,
        }
        return self._raw_get("wordlist", params)

    # ----- Fundament fuer d/e/f (Methoden vorhanden, Tasks folgen) ---------

    def concordance(
        self,
        corpname: str,
        q: str,
        usesubcorp: Optional[str] = None,
        pagesize: int = 1,
        asyn: int = 0,  # 0 = vollstaendige Ergebnisse (API-konform, keine Teildaten)
    ) -> Dict[str, Any]:
        params = {
            "corpname": corpname,
            "q": q,
            "usesubcorp": usesubcorp or None,
            "pagesize": pagesize,
            "asyn": asyn,
        }
        return self._get_with_poll("concordance", params, payload_key="Lines")

    def collx(
        self,
        corpname: str,
        q: str,
        cfromw: int = DEFAULTS["coll_window"][0],
        ctow: int = DEFAULTS["coll_window"][1],
        cminfreq: int = 5,
        cminbgr: int = 5,
        cbgrfns: Optional[List[str]] = None,
        cmaxitems: int = DEFAULTS["n_collocates"],
        cattr: str = "lemma",
        usesubcorp: Optional[str] = None,
    ) -> Dict[str, Any]:
        # VERIFIED: cbgrfns wird als konkatenierte Code-Zeichenkette erwartet
        # (z.B. "dmt"), NICHT als JSON-Array - sonst werden die JSON-Zeichen
        # als einzelne (Pseudo-)Codes interpretiert.
        codes = "".join(cbgrfns or DEFAULTS["cbgrfns"])
        params = {
            "corpname": corpname,
            "q": q,
            "cfromw": cfromw,
            "ctow": ctow,
            "cminfreq": cminfreq,
            "cminbgr": cminbgr,
            "cbgrfns": codes,
            "cmaxitems": cmaxitems,
            "cattr": cattr,
            "usesubcorp": usesubcorp or None,
        }
        return self._raw_get("collx", params)

    def extract_keywords(
        self,
        corpname: str,
        ref_corpname: str,
        usesubcorp: Optional[str] = None,
        simple_n: Optional[str] = None,   # OFFEN: Default wird beim Bau von e gesetzt
        minfreq: Optional[int] = None,
        maxfreq: Optional[int] = None,
        max_keywords: int = 100,
        wlblacklist: Optional[str] = None,
        usengrams: int = 0,
        ngrams_n: Optional[int] = None,
        ngrams_max_n: Optional[int] = None,
    ) -> Dict[str, Any]:
        params = {
            "corpname": corpname,
            "ref_corpname": ref_corpname,
            "usesubcorp": usesubcorp or None,
            "simple_n": simple_n,
            "minfreq": minfreq,
            "maxfreq": maxfreq,
            "max_keywords": max_keywords,
            "wlblacklist": wlblacklist,
            "usengrams": usengrams,
            "ngrams_n": ngrams_n,
            "ngrams_max_n": ngrams_max_n,
        }
        return self._raw_get("extract_keywords", params)


# ============================================================================
# SubcorpusManager  (Grundlage fuer e/f - Diskursmodus)
# ============================================================================

class SubcorpusManager:
    """
    Kapselt die andersartige, zustandsbehaftete Diskurs-Pipeline:
    Query-Subkorpus anlegen -> pollen -> nutzen -> (optional) loeschen.

    extract_keywords nimmt KEINE Query, sondern nur 'usesubcorp' (einen
    BENANNTEN Subkorpus). Das Subkorpus wird daher zuerst angelegt und
    bleibt persistent im Account erhalten (Default-Politik). Benennung:
    auto_disc_<lemma>_<JJJJMMTT>.
    """

    PREFIX = "auto_disc_"

    def __init__(self, client: SEClient, corpname: str):
        self.client = client
        self.corpname = corpname

    def make_name(self, lemma: str) -> str:
        return f"{self.PREFIX}{_sanitize(lemma)}_{_dt.datetime.now():%Y%m%d}"

    @staticmethod
    def cql_for_lemma(lemma: str, lpos: Optional[str] = None,
                      match_attr: str = "smart") -> str:
        """
        CQL fuer 'enthaelt Wort/Phrase'. Ein Token -> [...] (+ optional & tag).
        Mehrwort (Leerzeichen) -> Tokenfolge [...][...]. match_attr:
          'smart' (Default) -> [lc="x" | lemma_lc="x"], case-insensitiv auf
                               Wortform ODER Lemma (SE-Simple-Search-Idiom;
                               robust fuer gemischte Listen aus Einzelwoertern
                               und Phrasen, Eigennamen wie flektierten Formen).
          'lemma' / 'word'  -> exaktes, case-sensitives Matching auf dem Attribut.
          'lc'              -> case-insensitives Wortform-Matching.
        Bei Mehrwort wird lpos ignoriert (Phrasen-POS ist mehrdeutig).
        """
        tokens = lemma.split()

        def cond(tok: str) -> str:
            if match_attr == "smart":
                low = tok.lower().replace('"', '\\"')
                return f'lc="{low}" | lemma_lc="{low}"'
            if match_attr == "lc":
                return f'lc="{tok.lower().replace(chr(34), chr(92) + chr(34))}"'
            return f'{match_attr}="{tok.replace(chr(34), chr(92) + chr(34))}"'

        def one(tok: str, with_pos: bool) -> str:
            c = cond(tok)
            if with_pos and lpos:
                return f'[({c}) & tag="{lpos.lstrip("-")}.*"]'
            return f'[{c}]'

        if len(tokens) <= 1:
            return one(tokens[0] if tokens else lemma, with_pos=True)
        return "".join(one(t, with_pos=False) for t in tokens)

    def exists(self, subcname: str) -> bool:
        names = {s["name"] for s in self.client.list_subcorpora(self.corpname)}
        return subcname in names

    def create_query_subcorpus(
        self,
        subcname: str,
        q: str,
        struct: str = "doc",
        wait: bool = True,
    ) -> Optional[int]:
        """
        Legt ein Subkorpus aus einer Konkordanz-Query an (q + struct).
        struct='doc' => ganze Dokumente, in denen die Query trifft.
        VERIFIED: Anlage ist synchron ('Subcorpus saved.'), Groesse via
        subcorp_info.subcsize. Gibt die Subkorpusgroesse (Token) zurueck,
        bzw. None, wenn wait=False.
        """
        if self.exists(subcname):
            self.client._log(f"  [subcorp] '{subcname}' existiert bereits - wiederverwendet.")
            return self.wait_ready(subcname) if wait else None

        # 'q' fuer die Subkorpus-Anlage erwartet i.d.R. das fuehrende 'q'
        q_full = q if q.startswith("q") else "q" + q
        params = {
            "corpname": self.corpname,
            "subcname": subcname,
            "create": 1,
            "q": q_full,
            "struct": struct,
        }
        resp = self.client._raw_get("subcorp", params)
        msg = resp.get("message") if isinstance(resp, dict) else None
        self.client._log(f"  [subcorp] Anlage: '{subcname}' (struct={struct})"
                         + (f" - {msg}" if msg else ""))
        return self.wait_ready(subcname) if wait else None

    AUTO_PREFIX = "auto_disc_"

    def delete(self, subcname: str, allow_non_auto: bool = False) -> bool:
        """
        Loescht eine Subkorpus-DEFINITION via subcorp?delete=1. Laut SE-Doku wird
        dabei nur die Definition entfernt, keine Korpusdaten; das Subkorpus ist
        aus derselben Query jederzeit neu erzeugbar. Schutz: standardmaessig nur
        selbst erzeugte 'auto_disc_'-Subkorpora (allow_non_auto=True hebt das auf).
        Gibt True bei Erfolg zurueck.
        """
        if not allow_non_auto and not subcname.startswith(self.AUTO_PREFIX):
            self.client._log(f"  [subcorp] Loeschen abgelehnt: '{subcname}' ist kein "
                             f"'{self.AUTO_PREFIX}'-Subkorpus (Schutz).")
            return False
        try:
            resp = self.client._raw_get(
                "subcorp", {"corpname": self.corpname, "subcname": subcname, "delete": 1})
            msg = (resp.get("message") if isinstance(resp, dict) else None) or "geloescht"
            self.client._log(f"  [subcorp] '{subcname}' geloescht (nur Definition) - {msg}")
            return True
        except SketchEngineError as e:
            self.client._log(f"  [subcorp] Loeschen von '{subcname}' fehlgeschlagen: {e}")
            return False


    def wait_ready(self, subcname: str) -> Optional[int]:
        """
        Pollt subcorp_info, bis eine valide Groesse vorliegt. VERIFIED: die
        Subkorpusgroesse steht unter 'subcsize' (flach), nicht unter 'sizes'.
        Fertig = Schluessel vorhanden (auch subcsize=0 = leeres Subkorpus, dann
        Warnung). Harter Timeout -> Fehler. Anlage ist i.d.R. synchron
        ('Subcorpus saved.'), sodass der erste Aufruf schon greift.
        """
        deadline = time.time() + self.client.poll_timeout
        while True:
            try:
                info = self.client._raw_get(
                    "subcorp_info",
                    {"corpname": self.corpname, "subcname": subcname},
                )
                size = _to_int(_first_present(
                    info, ["subcsize", "size", "tokencount", "cnt", "wordcount"]))
                if size is not None:           # Schluessel da -> fertig
                    if size == 0:
                        self.client._log(
                            f"  [subcorp] '{subcname}' ist LEER (subcsize=0) - "
                            f"Zielwort evtl. nicht im Korpus.")
                    else:
                        self.client._log(f"  [subcorp] '{subcname}' bereit ({size} Token).")
                    return size
            except SketchEngineError:
                pass  # noch nicht abfragbar -> weiter pollen
            if time.time() >= deadline:
                raise SketchEngineTimeout(
                    f"Subkorpus '{subcname}' nicht fertig nach "
                    f"{self.client.poll_timeout:.0f}s. Abbruch."
                )
            self.client._log(f"  [subcorp] '{subcname}' wird erstellt; warte ...")
            time.sleep(self.client.poll_interval)


# ============================================================================
# Aufgabe c - Word Sketch
# ============================================================================

def _replace_w(relname: str, lemma: str) -> str:
    """Ersetzt den '%w'-Platzhalter im Relationsnamen durch das Zielwort."""
    if not isinstance(relname, str):
        return str(relname)
    return relname.replace("%w", lemma)


def task_c_word_sketch(
    client: SEClient,
    corpname: str,
    targets: List[Dict[str, str]],
    usesubcorp: Optional[str] = None,
    maxitems: int = DEFAULTS["wsketch_maxitems"],
    clustercolls: int = DEFAULTS["clustercolls"],
    relation_filter: Optional[List[str]] = None,
) -> "pd.DataFrame":
    """
    Word Sketch je Zielwort. Iteriert ueber die Zielwortliste (kein
    getrennter Lauf pro Wort fuer den Aufrufer). Liefert ein tidy
    Long-Format-DataFrame mit Spalte 'lemma'.

    Verifizierte Struktur: Gramrels[ ].Words[ ] mit word/count/score;
    erste Relation 'Constructions' herausgefiltert; '%w' im Relationsnamen
    beruecksichtigt. count = Kookkurrenz, score = logDice (Primaersortierung).
    """
    _ensure_deps()
    rows: List[Dict[str, Any]] = []
    stamp = _now_iso()
    focus = usesubcorp or "(Gesamtkorpus)"

    for t in targets:
        lemma = t["lemma"]
        lpos = t.get("lpos") or None
        if " " in lemma.strip():
            client._log(f"[c] uebersprungen (Mehrwort): '{lemma}' - Word Sketch ist single-lemma.")
            rows.append(_empty_row(lemma, lpos, corpname, focus, stamp,
                                   note="Mehrworteinheit - Word Sketch nicht anwendbar"))
            continue
        client._log(f"[c] Word Sketch: {lemma}{(' ' + lpos) if lpos else ''}")
        try:
            data = client.wsketch(
                corpname, lemma, lpos=lpos, usesubcorp=usesubcorp,
                maxitems=maxitems, clustercolls=clustercolls,
            )
        except SketchEngineError as exc:
            client._log(f"  ! Fehler bei '{lemma}': {exc}")
            rows.append(_empty_row(lemma, lpos, corpname, focus, stamp, note=str(exc)))
            continue

        gramrels = data.get("Gramrels") or []
        kept = 0
        for rel in gramrels:
            relname = _first_present(rel, ["name", "n"], "")
            # 'Constructions' ist Verteilungsstatistik, kein Kollokat -> filtern
            if isinstance(relname, str) and relname.strip().lower().startswith("construction"):
                continue
            if relation_filter and not _matches_filter(relname, relation_filter, lemma):
                continue
            words = _first_present(rel, ["Words", "words"], []) or []
            for w in words:
                rows.append({
                    "lemma": lemma,
                    "lpos": lpos or "",
                    "gramrel": _replace_w(relname, lemma),
                    "gramrel_raw": relname,
                    "collocate": _first_present(w, ["word", "lemma", "str"]),
                    "count": _to_int(_first_present(w, ["count", "freq", "cnt"])),
                    "logdice": _first_present(w, ["score", "logdice"]),
                    "corpus": corpname,
                    "focus": focus,
                    "timestamp": stamp,
                })
                kept += 1
        if kept == 0:
            client._log(f"  (leer) '{lemma}': keine Kollokate ueber Schwelle.")
            rows.append(_empty_row(lemma, lpos, corpname, focus, stamp,
                                   note="keine Kollokate (Beleglage/Schwelle)"))

    df = pd.DataFrame(rows)
    # logDice als Primaersortierung
    if not df.empty and "logdice" in df.columns:
        df = df.sort_values(
            ["lemma", "gramrel", "logdice"], ascending=[True, True, False],
            na_position="last",
        ).reset_index(drop=True)
    return df


def _matches_filter(relname: str, wanted: List[str], lemma: str) -> bool:
    """Relationsauswahl: vergleicht roh und mit ersetztem %w, case-insensitiv."""
    cand = {str(relname).lower(), _replace_w(str(relname), lemma).lower()}
    return any(w.lower() in cand or w.lower() in str(relname).lower() for w in wanted)


def _empty_row(lemma, lpos, corpname, focus, stamp, note=""):
    return {
        "lemma": lemma, "lpos": lpos or "", "gramrel": None, "gramrel_raw": None,
        "collocate": None, "count": None, "logdice": None,
        "corpus": corpname, "focus": focus, "timestamp": stamp, "note": note,
    }


# ============================================================================
# Kanal Z - Komposita / Wortbildung
# ============================================================================

def task_Z_compounds(
    client: SEClient,
    corpname: str,
    lemmas: List[str],
    usesubcorp: Optional[str] = None,
    wlminfreq: int = 5,
    wlmaxitems: int = DEFAULTS["wl_maxitems"],
    wlicase: int = DEFAULTS["wlicase"],
) -> "pd.DataFrame":
    """
    Komposita via wordlist (wlattr=lemma). Zwei Regex je Basis-Lemma:
        Determinans-Position:  <lemma>.+   (z.B. Tisch -> Tischdecke)
        Kopf-Position:         .+<lemma>   (z.B. Tisch -> Schreibtisch)
    Liefert tidy Long-Format mit Spalten lemma/position/compound/freq.
    Das Basis-Lemma selbst wird durch die '.+' automatisch ausgeschlossen.

    VERIFIED: Das Inline-Flag (?i) aus der Spezifikation matcht in SE NICHT
    (wird woertlich genommen -> 0 Treffer). Gross-/Kleinschreibung wird daher
    ueber wlicase gesteuert (Default 0 = case-insensitiv; so matcht .+Kraft
    auch 'Schwerkraft' mit kleinem k im Wortinneren).
    """
    _ensure_deps()
    rows: List[Dict[str, Any]] = []
    stamp = _now_iso()
    focus = usesubcorp or "(Gesamtkorpus)"

    positions = {
        "determinans": "{}.+",   # Lemma am Anfang
        "kopf": ".+{}",          # Lemma am Ende
    }

    for lemma in lemmas:
        if " " in lemma.strip():
            client._log(f"[Z] uebersprungen (Mehrwort): '{lemma}' - Wortbildung ist single-lemma.")
            rows.append({
                "lemma": lemma, "position": None, "compound": None, "freq": None,
                "corpus": corpname, "focus": focus, "timestamp": stamp,
                "note": "Mehrworteinheit - Komposita-Suche nicht anwendbar",
            })
            continue
        esc = re.escape(lemma)
        for position, pat_tpl in positions.items():
            wlpat = pat_tpl.format(esc)
            client._log(f"[Z] Komposita ({position}): {lemma}  pat={wlpat}")
            try:
                data = client.wordlist(
                    corpname, wlattr="lemma", wlpat=wlpat,
                    wlminfreq=wlminfreq, wlmaxitems=wlmaxitems,
                    wlicase=wlicase, usesubcorp=usesubcorp,
                )
            except SketchEngineError as exc:
                client._log(f"  ! Fehler bei '{lemma}'/{position}: {exc}")
                rows.append({
                    "lemma": lemma, "position": position, "compound": None,
                    "freq": None, "corpus": corpname, "focus": focus,
                    "timestamp": stamp, "note": str(exc),
                })
                continue

            items = _first_present(data, ["Items", "items", "blocks"], []) or []
            kept = 0
            for it in items:
                comp = _first_present(it, ["str", "word", "lemma"])
                if not comp or comp == lemma:   # Basiswort ausschliessen
                    continue
                rows.append({
                    "lemma": lemma,
                    "position": position,
                    "compound": comp,
                    "freq": _to_int(_first_present(it, ["frq", "freq", "count"])),
                    "corpus": corpname,
                    "focus": focus,
                    "timestamp": stamp,
                })
                kept += 1
            if kept == 0:
                client._log(f"  (leer) '{lemma}'/{position}: keine Treffer.")
                rows.append({
                    "lemma": lemma, "position": position, "compound": None,
                    "freq": None, "corpus": corpname, "focus": focus,
                    "timestamp": stamp, "note": "keine Treffer ueber Schwelle",
                })

    df = pd.DataFrame(rows)
    if not df.empty and "freq" in df.columns:
        df = df.sort_values(
            ["lemma", "position", "freq"], ascending=[True, True, False],
            na_position="last",
        ).reset_index(drop=True)
    return df


# ============================================================================
# Aufgabe d - Fensterkollokation (collx)
# ============================================================================

# cbgrfns-Code -> Spaltenname (vgl. OpenAPI 052_cbgrfns)
_CBGR_COL = {
    "d": "logdice", "m": "mi", "t": "tscore", "3": "mi3", "l": "loglik",
    "s": "minsens", "p": "mi_logf", "r": "relfreq", "f": "absfreq",
}


def task_d_window_collocation(
    client: SEClient,
    corpname: str,
    targets: List[Dict[str, str]],
    usesubcorp: Optional[str] = None,
    cfromw: int = DEFAULTS["coll_window"][0],
    ctow: int = DEFAULTS["coll_window"][1],
    cminfreq: int = 5,
    cminbgr: int = 5,
    cbgrfns: Optional[List[str]] = None,
    cmaxitems: int = DEFAULTS["n_collocates"],
    cattr: str = "lemma",
    match_attr: str = "lemma",
) -> "pd.DataFrame":
    """
    Fensterkollokation je Zielwort via collx (Query direkt als q). Iteriert
    ueber die Zielwortliste; Mehrworteinheiten werden als Tokenfolge gesucht.
    Fenster cfromw..ctow (Default -5..+5), cminfreq = Korpusfrequenz-Schwelle,
    cminbgr = Kookkurrenz im Fenster. Masse: logDice (Primaersortierung) + MI
    + T-Score. match_attr steuert das Knoten-Matching (lemma/word/lc).
    """
    _ensure_deps()
    cbgrfns = cbgrfns or DEFAULTS["cbgrfns"]   # ["d","m","t"]
    stat_cols = [_CBGR_COL.get(c, c) for c in cbgrfns]
    rows: List[Dict[str, Any]] = []
    stamp = _now_iso()
    focus = usesubcorp or "(Gesamtkorpus)"
    window = f"{cfromw}..{ctow}"

    for t in targets:
        lemma = t["lemma"]
        lpos = t.get("lpos") or None
        cql = SubcorpusManager.cql_for_lemma(lemma, lpos, match_attr=match_attr)
        q_full = "q" + cql                                  # -> q[...]
        client._log(f"[d] Fensterkollokation: {lemma} {window}  q={cql}")
        try:
            data = client.collx(
                corpname, q_full, cfromw=cfromw, ctow=ctow,
                cminfreq=cminfreq, cminbgr=cminbgr, cbgrfns=cbgrfns,
                cmaxitems=cmaxitems, cattr=cattr, usesubcorp=usesubcorp,
            )
        except SketchEngineError as exc:
            client._log(f"  ! Fehler bei '{lemma}': {exc}")
            rows.append({"lemma": lemma, "lpos": lpos or "", "collocate": None,
                         "freq": None, "window": window, "corpus": corpname,
                         "focus": focus, "timestamp": stamp, "note": str(exc)})
            continue

        items = _first_present(data, ["Items", "items"], []) or []
        kept = 0
        for it in items:
            row: Dict[str, Any] = {
                "lemma": lemma,
                "lpos": lpos or "",
                "collocate": _first_present(it, ["str", "word", "lemma"]),
                "freq": _to_int(_first_present(it, ["freq", "frq"])),
                "coll_freq": _to_int(_first_present(it, ["coll_freq", "collfreq"])),
            }
            # VERIFIED: Stats ist eine Liste {s: Wert, n: Code}. Zuordnung ueber
            # den NAMEN n (robust gegen Reihenfolge und gegen Pseudo-Eintraege).
            stats = it.get("Stats")
            if isinstance(stats, list):
                for st in stats:
                    if not isinstance(st, dict):
                        continue
                    col = _CBGR_COL.get(st.get("n"))
                    if col:
                        row[col] = _to_float(st.get("s"))
            else:
                # Fallback: benannte Felder oder Kuerzel direkt am Item
                for code, col in zip(cbgrfns, stat_cols):
                    row[col] = _to_float(_first_present(it, [col, code]))
            row.update({"window": window, "corpus": corpname,
                        "focus": focus, "timestamp": stamp})
            rows.append(row)
            kept += 1
        if kept == 0:
            client._log(f"  (leer) '{lemma}': keine Kollokate ueber Schwelle.")
            rows.append({"lemma": lemma, "lpos": lpos or "", "collocate": None,
                         "freq": None, "window": window, "corpus": corpname,
                         "focus": focus, "timestamp": stamp,
                         "note": "keine Kollokate (Beleglage/Schwelle)"})

    df = pd.DataFrame(rows)
    if not df.empty and "logdice" in df.columns:
        df = df.sort_values(
            ["lemma", "logdice"], ascending=[True, False], na_position="last",
        ).reset_index(drop=True)
    return df


# ============================================================================
# Aufgaben e/f - Stubs (Fundament steht; Tasks folgen im naechsten Schritt)
# ============================================================================

# ============================================================================
# Aufgabe e - Schluesselwoerter (extract_keywords) + Diskursmodus
# ============================================================================

def task_e_keywords(
    client: SEClient,
    corpname: str,
    ref_corpname: str,
    node_lemma: Optional[str] = None,
    usesubcorp: Optional[str] = None,
    simple_n: str = "5",
    minfreq: int = 5,
    max_keywords: int = 100,
    wlblacklist: Optional[str] = None,
    usengrams: int = 0,
    ngrams_n: Optional[int] = None,
    ngrams_max_n: Optional[int] = None,
) -> "pd.DataFrame":
    """
    Ein extract_keywords-Lauf -> tidy DataFrame. Fokus = corpname (+ usesubcorp),
    Referenz = ref_corpname. 'node_lemma' ist im Diskursmodus das Zielwort (wird
    als Spalte 'lemma' gefuehrt, damit die Pro-Lexem-Ausgabe greift); im
    Plain-Modus None.

    VERIFY (im e-Smoke-Test): extract_keywords-Antwortstruktur (hier defensiv
    geparst); ob simple_n Werte > 1 akzeptiert (OpenAPI nennt nur {0,1}).
    """
    _ensure_deps()
    stamp = _now_iso()
    data = client.extract_keywords(
        corpname, ref_corpname, usesubcorp=usesubcorp, simple_n=simple_n,
        minfreq=minfreq, max_keywords=max_keywords, wlblacklist=wlblacklist,
        usengrams=usengrams, ngrams_n=ngrams_n, ngrams_max_n=ngrams_max_n,
    )
    items = _first_present(data, ["keywords", "Items", "data", "items"], []) or []
    rows: List[Dict[str, Any]] = []
    for it in items:
        rows.append({
            "lemma": node_lemma,                       # Diskurswort (oder None)
            "keyword": _first_present(it, ["item", "str", "word", "lemma"]),
            "keyness": _to_float(_first_present(it, ["score", "keyness"])),
            "freq_focus": _to_int(_first_present(it, ["frq1", "freq", "frq"])),
            "freq_ref": _to_int(_first_present(it, ["frq2", "ref_freq", "frq_ref"])),
            "relfreq_focus": _to_float(_first_present(it, ["rel_frq1", "relfreq"])),
            "relfreq_ref": _to_float(_first_present(it, ["rel_frq2"])),
            "query": _first_present(it, ["query"]),
            "corpus": corpname,
            "focus_subcorpus": usesubcorp or "(Gesamtkorpus)",
            "reference_corpus": ref_corpname,
            "simple_n": simple_n,
            "timestamp": stamp,
        })
    df = pd.DataFrame(rows)
    if not df.empty and "keyness" in df.columns:
        sort_cols = (["lemma", "keyness"] if df["lemma"].notna().any() else ["keyness"])
        asc = [True, False] if len(sort_cols) == 2 else [False]
        df = df.sort_values(sort_cols, ascending=asc, na_position="last").reset_index(drop=True)
    return df


def run_keywords(client: SEClient, cfg: Dict[str, Any], usengrams: int = 0,
                 on_partial=None) -> "pd.DataFrame":
    """
    Orchestriert Aufgabe e (und spaeter f via usengrams). Plain-Modus:
    Fokus(korpus/-subkorpus) vs. Referenz. Diskursmodus: je Zielwort ein
    Query-Subkorpus aller Dokumente mit dem Wort (struct=doc), dann Keywords
    gegen die Referenz; Knoten wird optional geblacklistet. Subkorpora bleiben
    erhalten (auto_disc_*).

    on_partial(df): optionaler Callback, nach JEDEM Zielwort mit dem bisherigen
    (gestapelten) Ergebnis aufgerufen - fuer Zwischenspeicherung durch run().
    """
    _ensure_deps()
    corpname = _cfg(cfg, "corpus", "corpname")
    ref = _cfg(cfg, "reference_corpus", "ref_corpname")
    simple_n = str(_cfg(cfg, "kw_simple_n", default="5"))
    minfreq = _cfg(cfg, "kw_min_freq", "kw_minfreq", default=5)
    max_keywords = _cfg(cfg, "kw_max_keywords", default=100)
    match_attr = _cfg(cfg, "node_match_attr", default="smart")
    targets = cfg.get("targets", [])
    discourse = bool(cfg.get("discourse_mode"))
    ng_kwargs = {"usengrams": usengrams,
                 "ngrams_n": cfg.get("ngrams_n", 2) if usengrams else None,
                 "ngrams_max_n": cfg.get("ngrams_max_n", 3) if usengrams else None}

    if not discourse or not targets:
        # Plain-Modus: ein Lauf gegen die Referenz (Fokus = optionales Subkorpus)
        client._log("[e] Schluesselwoerter (Plain-Modus, Fokus vs. Referenz)")
        return task_e_keywords(
            client, corpname, ref, node_lemma=None,
            usesubcorp=_cfg(cfg, "focus_subcorpus", "usesubcorp"),
            simple_n=simple_n, minfreq=minfreq, max_keywords=max_keywords, **ng_kwargs)

    # Diskursmodus
    sm = SubcorpusManager(client, corpname)
    struct = _cfg(cfg, "discourse_subcorpus_struct", "discourse_struct", default="doc")
    per_word = bool(_cfg(cfg, "discourse_subcorpus_per_word", "discourse_per_word", default=True))
    do_blacklist = bool(_cfg(cfg, "discourse_blacklist_node", "blacklist_node", default=True))
    corpus_size = _to_int(_cfg(cfg, "corpus_size_tokens", default=None))
    max_share = float(_cfg(cfg, "kw_max_subcorpus_share",
                            default=DEFAULTS["kw_max_subcorpus_share"]))
    share_action = str(_cfg(cfg, "kw_large_subcorpus_action",
                            default=DEFAULTS["kw_large_subcorpus_action"])).lower()
    delete_oversized = bool(_cfg(cfg, "kw_delete_oversized", default=False))
    ask_timeout = float(_cfg(cfg, "kw_ask_timeout", default=DEFAULTS["kw_ask_timeout"]))
    timed_out = {"v": False}   # nach erstem Timeout: weitere 'ask' automatisch -> Nein

    def too_large_skip(lemma: str, name: str, sub_size: Optional[int]) -> Optional["pd.DataFrame"]:
        """
        Prueft den Subkorpus-Anteil am Korpus. Ueber der Schwelle: je nach Politik
        ueberspringen / nachfragen / dennoch rechnen. Rueckgabe: Skip-DataFrame
        (1 Zeile mit Notiz) wenn uebersprungen wird, sonst None (-> weiterrechnen).
        Beim Ueberspringen wird das (auto_disc_) Subkorpus optional geloescht.
        """
        if not (corpus_size and sub_size and corpus_size > 0):
            return None
        share = sub_size / corpus_size
        if share <= max_share:
            return None
        client._log(f"  [e] '{lemma}': Subkorpus = {share:.1%} des Korpus (> {max_share:.0%}). "
                    f"Keyness gegen das Gesamtkorpus wird dadurch flach/wenig aussagekraeftig.")
        proceed = (share_action == "force")
        if share_action == "ask":
            if timed_out["v"]:
                client._log("        -> (nach vorheriger Zeitueberschreitung) automatisch uebersprungen.")
                proceed = False
            else:
                try:
                    ans = _ask_yes_no_timeout(
                        f"        Trotzdem Keywords fuer '{lemma}' berechnen?", ask_timeout)
                    if ans is None:                    # Timeout
                        client._log(f"        -> keine Antwort in {int(ask_timeout)}s "
                                    f"-> uebersprungen (Default Nein).")
                        timed_out["v"] = True
                        proceed = False
                    else:
                        proceed = ans
                except (EOFError, OSError):
                    client._log("        -> keine Eingabe moeglich (headless) -> uebersprungen.")
                    proceed = False
        if proceed:
            client._log("        -> wird dennoch berechnet.")
            return None
        client._log("        -> uebersprungen.")
        deleted = False
        if delete_oversized:
            deleted = sm.delete(name)            # nur auto_disc_, nur Definition
        return pd.DataFrame([{
            "lemma": lemma, "keyword": None,
            "note": (f"uebersprungen: Subkorpus {share:.1%} des Korpus > {max_share:.0%} "
                     f"(Keyness wenig aussagekraeftig)"
                     + ("; Subkorpus-Definition geloescht" if deleted else "")),
        }])

    def effective_minfreq(sub_size: Optional[int]) -> int:
        # minfreq an die SUBKORPUSGROESSE anpassen, nicht an die Gesamtkorpusgroesse.
        # Sonst filtert eine aus dem Milliarden-Korpus abgeleitete Schwelle (z.B. 100)
        # die Schluesselwoerter eines kleinen Diskurs-Subkorpus komplett weg.
        if not isinstance(sub_size, int) or sub_size <= 0:
            return minfreq
        sub_default = size_tier_defaults(sub_size)["kw_minfreq"]
        eff = min(minfreq, sub_default)
        if eff != minfreq:
            client._log(f"  [e] minfreq an Subkorpus angepasst: {minfreq} -> {eff} "
                        f"(Subkorpus {sub_size} Token)")
        return eff

    def blacklist_for(lemma: str) -> Optional[str]:
        # Mehrwort -> Bestandteile einzeln blacklisten. Keywords sind auf 'lc'
        # (lowercased), daher jeden Bestandteil in Klein- UND Originalschreibung,
        # sonst steht der Knoten trivial als eigenes Top-Keyword oben.
        if not do_blacklist:
            return None
        entries: List[str] = []
        for tok in lemma.split():
            for form in (tok, tok.lower()):
                if form not in entries:
                    entries.append(form)
        return "\n".join(entries)

    ready_timeout = float(_cfg(cfg, "kw_ready_timeout", default=DEFAULTS["kw_ready_timeout"]))
    ready_wait = float(_cfg(cfg, "kw_ready_wait", default=DEFAULTS["kw_ready_wait"]))

    def keywords_ready(lemma, name, sub_size, bl):
        """
        Ruft task_e_keywords; falls 0 Treffer trotz nicht-leerem Subkorpus
        (subcsize>0), wartet und versucht erneut, bis Treffer kommen oder das
        Zeitbudget erschoepft ist. Frisch angelegte grosse Subkorpora sind erst
        nach SE-seitiger Kompilierung abfragebereit.
        """
        def call():
            return task_e_keywords(
                client, corpname, ref, node_lemma=lemma, usesubcorp=name,
                simple_n=simple_n, minfreq=effective_minfreq(sub_size),
                max_keywords=max_keywords, wlblacklist=bl, **ng_kwargs)

        df = call()
        if not df.empty or not (isinstance(sub_size, int) and sub_size > 0):
            return df  # Treffer, oder Subkorpus echt leer -> nicht warten
        deadline = time.time() + ready_timeout
        wait = ready_wait
        attempt = 1
        while df.empty and time.time() < deadline:
            client._log(f"  [e] '{lemma}': 0 Treffer bei subcsize={sub_size} - Subkorpus "
                        f"wird vermutlich noch kompiliert; warte {wait:.0f}s, neuer Versuch "
                        f"(#{attempt + 1}) ...")
            time.sleep(wait)
            df = call()
            attempt += 1
            wait = min(wait * 1.5, 30.0)
        if df.empty:
            client._log(f"  [e] '{lemma}': nach {ready_timeout:.0f}s weiterhin 0 Treffer. "
                        f"Subkorpus evtl. noch nicht bereit - spaeter erneut laufen lassen "
                        f"(es wird wiederverwendet) oder kw_ready_timeout erhoehen.")
        else:
            client._log(f"  [e] '{lemma}': abfragebereit, {len(df)} Treffer.")
        return df

    if per_word:
        dfs: List["pd.DataFrame"] = []
        for t in targets:
            lemma, lpos = t["lemma"], t.get("lpos") or None
            name = sm.make_name(lemma)
            q = sm.cql_for_lemma(lemma, lpos, match_attr=match_attr)
            client._log(f"[e] Diskurs '{lemma}': Subkorpus '{name}' (struct={struct})")
            sub_size = sm.create_query_subcorpus(name, q, struct=struct, wait=True)
            skip = too_large_skip(lemma, name, sub_size)
            if skip is not None:
                dfs.append(skip)
            else:
                dfs.append(keywords_ready(lemma, name, sub_size, blacklist_for(lemma)))
            if on_partial is not None:
                on_partial(pd.concat(dfs, ignore_index=True))
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    # Union-Modus: ein Subkorpus aus OR aller Phrasen, ein Keyword-Lauf
    lemmas = [t["lemma"] for t in targets]
    name = sm.make_name("union_" + "_".join(_sanitize(l) for l in lemmas))[:60]
    # OR ueber ganze (auch mehrwortige) Phrasen via Alternativ-Bloecke
    cql = "|".join(sm.cql_for_lemma(l, match_attr=match_attr) for l in lemmas)
    client._log(f"[e] Diskurs (Union {lemmas}): Subkorpus '{name}'")
    sub_size = sm.create_query_subcorpus(name, cql, struct=struct, wait=True)
    skip = too_large_skip("(union)", name, sub_size)
    if skip is not None:
        return skip
    bl = ("\n".join(dict.fromkeys(
        form for l in lemmas for tok in l.split() for form in (tok, tok.lower())))
        if do_blacklist else None)
    return keywords_ready("(union)", name, sub_size, bl)


# ============================================================================
# Aufgabe f - Mehrwort-Schluesseleinheiten (extract_keywords + usengrams)
# ============================================================================

def task_f_multiword_keys(client: SEClient, cfg: Dict[str, Any]) -> "pd.DataFrame":
    """
    Mehrwort-Schluesseleinheiten = Aufgabe e mit n-Grammen. Duenner Wrapper um
    run_keywords(usengrams=1); teilt die gesamte Plain-/Diskurs-/Subkorpus-Logik
    mit e. Laenge ueber ngrams_n / ngrams_max_n.

    VERIFY (im f-Smoke-Test): ob die n-Gramm-Items in derselben 'keywords'-
    Struktur zurueckkommen (erwartet: ja, mit mehrwortigem 'item').
    """
    return run_keywords(client, cfg, usengrams=1)


# ============================================================================
# Config - interaktiv, bedarfsgesteuert, als JSON gespeichert
# ============================================================================

def _ask(prompt: str, default: Optional[str] = None) -> str:
    suffix = f" [{default}]" if default is not None else ""
    val = input(f"{prompt}{suffix}: ").strip()
    return val or (default or "")


def _ask_yes_no(prompt: str, default: bool = False) -> bool:
    d = "J/n" if default else "j/N"
    val = input(f"{prompt} [{d}]: ").strip().lower()
    if not val:
        return default
    return val in ("j", "ja", "y", "yes")


def _ask_int(prompt: str, default: int) -> int:
    val = input(f"{prompt} [{default}]: ").strip()
    return _to_int(val, default) if val else default


def _ask_yes_no_timeout(prompt: str, timeout: float):
    """
    Ja/Nein-Abfrage mit Timeout. Rueckgabe: True (ja), False (nein) oder None bei
    Zeitueberschreitung (-> Aufrufer behandelt das als Default-Nein). Nutzt einen
    Daemon-Thread fuer input(); funktioniert im Terminal zuverlaessig und in
    Colab-Zellen im Normalfall. EOFError/OSError (headless) werden durchgereicht.
    """
    if not timeout or timeout <= 0:
        return input(f"{prompt} [j/N]: ").strip().lower() in ("j", "ja", "y", "yes")
    box: Dict[str, Any] = {}

    def _read():
        try:
            box["v"] = input(f"{prompt} [j/N, {int(timeout)}s bis Nein]: ")
        except BaseException as e:   # EOFError/OSError (headless) o.ae.
            box["err"] = e

    th = threading.Thread(target=_read, daemon=True)
    th.start()
    th.join(timeout)
    if th.is_alive():
        return None                  # Timeout -> Aufrufer: Default-Nein
    if "err" in box:
        raise box["err"]
    return str(box.get("v", "")).strip().lower() in ("j", "ja", "y", "yes")


def _ask_multi(prompt: str, options: List[str]) -> List[str]:
    print(prompt)
    for o in options:
        label = TASKS[o][0] if o in TASKS else o
        print(f"   {o} = {label}")
    raw = input("   Auswahl (Kuerzel, komma-getrennt): ").strip()
    chosen = [x.strip() for x in re.split(r"[,\s]+", raw) if x.strip()]
    return [c for c in chosen if c in options]


def _parse_target_line(line: str) -> Optional[Dict[str, str]]:
    """
    Zielwort-Zeile -> {lemma, lpos}. Phrase bleibt intakt; ein abschliessendes
    POS-Tag (Token, das mit '-' beginnt) wird abgetrennt:
        'Tisch -n'                -> {'Tisch', '-n'}
        'Dominikanische Republik' -> {'Dominikanische Republik', ''}
        'elektrisches Feld -n'    -> {'elektrisches Feld', '-n'}
    """
    parts = line.strip().split()
    if not parts:
        return None
    lpos = ""
    if len(parts) >= 2 and parts[-1].startswith("-"):
        lpos = parts[-1]
        parts = parts[:-1]
    lemma = " ".join(parts).strip()
    if not lemma:
        return None
    return {"lemma": lemma, "lpos": lpos}


def _ask_targets(intro: bool = True) -> List[Dict[str, str]]:
    """Sammelt Zielwoerter zeilenweise (Phrase intakt, optionales -POS); leere Zeile beendet."""
    if intro:
        print("Zielwoerter (eine Zeile je Eintrag); leere Zeile beendet.")
        print("  Einzelwort optional mit POS am Ende: 'Tisch -n'")
        print("  Mehrworteinheit einfach als Phrase: 'Dominikanische Republik'")
        print("  (Mehrwort wirkt nur fuer d/e/f; c und Z ueberspringen Phrasen.)")
    out: List[Dict[str, str]] = []
    while True:
        line = input("   > ").strip()
        if not line:
            break
        p = _parse_target_line(line)
        if p:
            out.append(p)
    return out


def _override_loaded_config(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """
    Erlaubt nach dem Laden einer Config das gezielte UEBERSCHREIBEN einzelner
    Felder bei sonst gleichen Parametern - v.a. der Zielwoerter ('gleiche
    Einstellungen, andere Lemmata'). Korpuswechsel wird hier bewusst NICHT
    angeboten: corpus_size_tokens und die groessenabhaengigen Defaults (minfreq
    etc.) haengen am Korpus; dafuer ist ein frischer interaktiver Lauf sauberer.
    """
    tgt = cfg.get("targets") or []
    print(f"  Geladen: Korpus={cfg.get('corpus')} | Aufgaben={cfg.get('tasks')} | "
          f"Zielwoerter={[t.get('lemma') for t in tgt]}")
    if not _ask_yes_no("Etwas an der geladenen Config aendern (sonst 1:1 uebernehmen)?",
                       default=False):
        return cfg

    # Zielwoerter neu setzen (Hauptfall)
    if "targets" in cfg and _ask_yes_no(
            "Zielwoerter neu eingeben (gleiche Parameter, andere Lemmata)?", default=True):
        old_lemmas = [t.get("lemma") for t in tgt]
        new = _ask_targets()
        if new:
            cfg["targets"] = new
            # compound_lemmas mitziehen, falls sie an die Zielwortliste gekoppelt waren
            if cfg.get("compound_lemmas") == old_lemmas:
                cfg["compound_lemmas"] = [t["lemma"] for t in new]
            cfg["created"] = _now_iso()
        else:
            print("  (keine neuen Zielwoerter eingegeben - alte bleiben)")

    # Referenzkorpus (orthogonal zur Korpusgroesse, daher unbedenklich)
    if "reference_corpus" in cfg and _ask_yes_no(
            f"Referenzkorpus aendern (aktuell {cfg.get('reference_corpus')})?", default=False):
        cfg["reference_corpus"] = _ask("Referenzkorpus (ref_corpname)",
                                       cfg.get("reference_corpus"))

    # Ausgabe-Layout
    if _ask_yes_no(f"Ausgabe-Layout aendern (aktuell {cfg.get('output_layout','both')})?",
                   default=False):
        lay = _ask("Layout (both | by_task | by_lexeme)", cfg.get("output_layout", "both"))
        cfg["output_layout"] = lay if lay in ("both", "by_task", "by_lexeme") else cfg.get("output_layout", "both")

    return cfg


def build_config_interactive(client_probe: bool = True) -> Tuple[Dict[str, Any], SEClient]:
    """
    Baut die Config bedarfsgesteuert auf und gibt (config, client) zurueck.
    Reihenfolge: Replay-Weiche -> Auth -> Aufgaben -> Fokus(+corp_info live)
    -> bedarfsgesteuerte Eingaben -> Parameter mit Defaults -> Druck+Speichern.
    Der API-Key landet NICHT in der Config.
    """
    # --- Replay-Weiche ---
    path = _ask("Bestehende Config-JSON laden? (Pfad oder leer fuer interaktiv)", "")
    if path:
        cfg = load_config(path)
        cfg.pop("_template", None)
        user, key = resolve_credentials()
        client = SEClient(user, key,
                          throttle_seconds=cfg.get("throttle_seconds", DEFAULTS["throttle_seconds"]))
        # Template ohne Lemmata? -> Zielwoerter jetzt erfragen.
        if _targets_required(cfg) and not cfg.get("targets"):
            print("  Template erkannt (Config ohne Zielwoerter) - bitte Lemmata eingeben.")
            cfg["targets"] = _ask_targets()
            if "Z" in cfg.get("tasks", []) and not cfg.get("compound_lemmas"):
                cfg["compound_lemmas"] = [t["lemma"] for t in cfg["targets"]]
            cfg["created"] = _now_iso()
        cfg = _override_loaded_config(cfg)
        validate_config(cfg)
        print_config(cfg)
        return cfg, client

    # --- Auth (frueh, fail-fast) ---
    user, key = resolve_credentials()
    client = SEClient(user, key)

    # --- Aufgaben zuerst ---
    tasks = _ask_multi("Welche Aufgaben?", list(TASKS.keys()))
    while not tasks:
        print("Bitte mindestens eine Aufgabe waehlen.")
        tasks = _ask_multi("Welche Aufgaben?", list(TASKS.keys()))

    needs_node = any(TASKS[t][1] == "node" for t in tasks)
    needs_contrast = any(TASKS[t][1] == "contrast" for t in tasks)

    # --- Fokuskorpus + Live-Validierung via corp_info ---
    corpname = _ask("Fokuskorpus (z.B. user/.../x oder preloaded/...)")
    print("  -> pruefe Korpus (corp_info) ...")
    size = client.corpus_size(corpname)        # validiert Auth + Korpusname zugleich
    print(f"  -> Korpus ok: {_format_tokens(size)}")
    subcs = client.list_subcorpora(corpname)

    # --- Fokusmodus (global fuer c/d/Z UND e/f) ---
    focus_subcorpus = None
    if subcs:
        print("  Benannte Subkorpora:", ", ".join(s["name"] for s in subcs))
    if _ask_yes_no("Auf benanntes Subkorpus einschraenken?", default=False):
        focus_subcorpus = _ask("Subkorpus-Name (usesubcorp)") or None

    tiers = size_tier_defaults(size)

    cfg: Dict[str, Any] = {
        "created": _now_iso(),
        "username": user,                  # KEIN Key
        "tasks": tasks,
        "corpus": corpname,
        "corpus_size_tokens": size,
        "focus_subcorpus": focus_subcorpus,
        "throttle_seconds": DEFAULTS["throttle_seconds"],
    }

    # --- Bedarfsgesteuerte Eingaben ---
    # Zielwoerter: noetig fuer node-Aufgaben (c/d/Z)
    if needs_node:
        targets = _ask_targets()
        if not targets:
            raise ValueError("Knotenzentrierte Aufgaben gewaehlt, aber keine Zielwoerter.")
        cfg["targets"] = targets

        # Kanal Z: eigene Lemma-Liste? (Default: nein -> teilt Zielwortliste)
        if "Z" in tasks:
            if _ask_yes_no("Kanal Z: eigene Lemma-Liste fuer Komposita?", default=False):
                print("   Lemmata (eine Zeile je Wort); leere Zeile beendet:")
                zl: List[str] = []
                while True:
                    line = input("   > ").strip()
                    if not line:
                        break
                    zl.append(line.split()[0])
                cfg["compound_lemmas"] = zl
            else:
                cfg["compound_lemmas"] = [t["lemma"] for t in targets]

    # Referenzkorpus + Diskursmodus: noetig fuer contrast-Aufgaben (e/f)
    if needs_contrast:
        cfg["reference_corpus"] = _ask("Referenzkorpus (ref_corpname)")
        cfg["discourse_mode"] = _ask_yes_no(
            "Diskursmodus? (Subkorpus aller Dokumente mit dem Zielwort)", default=True
        )
        if cfg["discourse_mode"]:
            if not needs_node:
                # Diskurs braucht Zielwoerter, auch wenn keine node-Aufgabe gewaehlt
                cfg["targets"] = _ask_targets()
            cfg["discourse_subcorpus_struct"] = _ask("Dokument-Struktur fuer Subkorpus", "doc")
            cfg["discourse_subcorpus_per_word"] = _ask_yes_no(
                "Pro Zielwort ein eigenes Subkorpus? (sonst Union)", default=True
            )
            cfg["discourse_blacklist_node"] = _ask_yes_no(
                "Knoten vor Keyword-Berechnung ausschliessen (blacklisten)?", default=True
            )
            print("  Hinweis: SE vergleicht per Default gegen das Gesamtkorpus "
                  "inkl. Subkorpus (bei kleinem Subkorpus vernachlaessigbar).")
            share = _ask("Max. Subkorpus-Anteil am Korpus fuer Keywords (0-1; darueber "
                         "wird die Keyness flach). Empfehlung 0.05", "0.05")
            try:
                cfg["kw_max_subcorpus_share"] = max(0.0, min(1.0, float(share)))
            except ValueError:
                cfg["kw_max_subcorpus_share"] = DEFAULTS["kw_max_subcorpus_share"]
            act = _ask("Bei Ueberschreitung: ask (nachfragen) | skip (ueberspringen) | "
                       "force (trotzdem)", "ask").strip().lower()
            cfg["kw_large_subcorpus_action"] = act if act in ("ask", "skip", "force") else "ask"
            if cfg["kw_large_subcorpus_action"] != "force":
                cfg["kw_delete_oversized"] = _ask_yes_no(
                    "Uebergrosses auto-Subkorpus nach dem Ueberspringen loeschen? "
                    "(nur die Definition, jederzeit neu erzeugbar)", default=True)
        # simple_n bleibt OFFEN bis zum Bau von Aufgabe e.

    # --- Aufgabenparameter mit (groessenabhaengigen) Defaults ---
    if any(t in tasks for t in ("d", "e", "f")):
        ma = _ask("Knoten-Matching fuer d/e/f (smart | lemma | word | lc). "
                  "'smart' = lc|lemma_lc, robust fuer gemischte Listen aus Wort + "
                  "Phrase (Eigennamen wie flektierte Formen)", "smart").strip().lower()
        cfg["node_match_attr"] = ma if ma in ("smart", "lemma", "word", "lc") else "smart"
    if "c" in tasks:
        cfg["ws_max_items_per_relation"] = _ask_int(
            "c: max. Kollokate je grammatischer Relation (wsketch maxitems)",
            DEFAULTS["wsketch_maxitems"])
        cfg["ws_cluster_collocates"] = DEFAULTS["clustercolls"]
    if "d" in tasks:
        wl, wr = DEFAULTS["coll_window"]                     # (-5, 5)
        left = _ask_int("d: linke Fenstergrenze (Tokens links vom Knoten, negativ)", wl)
        right = _ask_int("d: rechte Fenstergrenze (Tokens rechts vom Knoten, positiv)", wr)
        if left > right:
            print(f"   Hinweis: linke Grenze {left} > rechte {right} - getauscht.")
            left, right = right, left
        if left == 0 and right == 0:
            print("   Hinweis: Fenster 0..0 erfasst nur den Knoten - auf -5..+5 zurueckgesetzt.")
            left, right = wl, wr
        cfg["coll_window"] = [left, right]
        cfg["coll_min_corpus_freq"] = _ask_int(
            "d: Mindest-Korpusfrequenz eines Kollokats (cminfreq, groessenabhaengig)",
            tiers["cminfreq"])
        cfg["coll_min_cooccurrence_freq"] = _ask_int(
            "d: Mindest-Kookkurrenz im Fenster (cminbgr, groessenabhaengig)",
            tiers["cminbgr"])
        cfg["coll_max_collocates"] = _ask_int(
            "d: max. Anzahl Kollokate je Zielwort (cmaxitems)", DEFAULTS["n_collocates"])
    if "Z" in tasks:
        cfg["comp_min_freq"] = _ask_int(
            "Z: Mindestfrequenz eines Kompositums (wlminfreq, groessenabhaengig)",
            tiers["wlminfreq"])
        cfg["comp_max_items"] = _ask_int(
            "Z: max. Anzahl Komposita je Position (wlmaxitems)", DEFAULTS["wl_maxitems"])
        cfg["comp_case_insensitive"] = True   # matcht auch '...kraft' im Wortinneren
    if "e" in tasks or "f" in tasks:
        cfg["kw_min_freq"] = _ask_int(
            "e/f: Mindestfrequenz eines Schluesselworts (minfreq, groessenabhaengig)",
            tiers["kw_minfreq"])
        cfg["kw_max_keywords"] = _ask_int(
            "e/f: max. Anzahl Schluesselwoerter (max_keywords)", 100)
        cfg["kw_simple_n"] = _ask(
            "e/f: simple_n (Keyness-Glaettung; 1=seltene/Terminologie, "
            "100=haeufige/robust)", "5")
    if "f" in tasks:
        cfg["ngrams_n"] = _ask_int("f: minimale n-Gramm-Laenge (ngrams_n)", 2)
        cfg["ngrams_max_n"] = _ask_int("f: maximale n-Gramm-Laenge (ngrams_max_n)", 3)

    # --- Ausgabe-Layout ---
    per_lexeme = _ask_yes_no(
        "Ausgabe zusaetzlich pro Lexem (eine Datei je Lemma, z.B. Tisch getrennt "
        "von Stuhl)?", default=True)
    keep_master = _ask_yes_no(
        "Tidy-Gesamttabelle behalten (alle Lexeme je Aufgabe in einem Workbook)?",
        default=True)
    cfg["output_layout"] = ("both" if (per_lexeme and keep_master)
                            else "by_lexeme" if per_lexeme else "by_task")

    print_config(cfg)
    return cfg, client


def print_config(cfg: Dict[str, Any]) -> None:
    """Druckt die Config OHNE Key (zur Kontrolle)."""
    safe = dict(cfg)
    print("\n=== Konfiguration (ohne API-Key) ===")
    print(json.dumps(safe, ensure_ascii=False, indent=2))
    print("====================================\n")


def save_config(cfg: Dict[str, Any], out_dir: str, filename: str = "run_config.json") -> str:
    """Speichert die Config als JSON (ohne Key)."""
    path = os.path.join(out_dir, filename)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(cfg, fh, ensure_ascii=False, indent=2)
    return path


# Felder, die ein wiederverwendbares Template NICHT enthalten soll (lemma-spezifisch).
_TEMPLATE_DROP = ("targets", "compound_lemmas")


def save_config_template(cfg: Dict[str, Any], out_dir: str, filename: str) -> str:
    """
    Speichert die Config OHNE Lemmata (targets/compound_lemmas) als wiederverwendbares
    Template. Beim Laden eines solchen Templates fragt das Werkzeug die Zielwoerter
    interaktiv ab - 'gleiche Parameter, andere Lemmata' ohne Handarbeit an der JSON.
    """
    tmpl = {k: v for k, v in cfg.items() if k not in _TEMPLATE_DROP}
    tmpl["_template"] = True   # Markierung: bewusst ohne Lemmata
    return save_config(tmpl, out_dir, filename)


def _targets_required(cfg: Dict[str, Any]) -> bool:
    """Zielwoerter noetig? Bei jeder knotenzentrierten Aufgabe (c/d/Z) oder bei
    korpuskontrastiven Aufgaben (e/f) im Diskursmodus."""
    tasks = cfg.get("tasks", []) or []
    if any(TASKS.get(t, (None, None))[1] == "node" for t in tasks):
        return True
    if any(TASKS.get(t, (None, None))[1] == "contrast" for t in tasks) and cfg.get("discourse_mode"):
        return True
    return False


def load_config(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


def validate_config(cfg: Dict[str, Any]) -> None:
    """Minimal-Schema fuer headless Replay-Laeufe."""
    if not isinstance(cfg, dict):
        raise ValueError("Config ist kein Objekt.")
    if not _cfg(cfg, "corpus", "corpname"):
        raise ValueError("Config: 'corpus' fehlt.")
    tasks = cfg.get("tasks")
    if not tasks or not all(t in TASKS for t in tasks):
        raise ValueError(f"Config: 'tasks' ungueltig (erlaubt: {list(TASKS)}).")
    node = any(TASKS[t][1] == "node" for t in tasks)
    contrast = any(TASKS[t][1] == "contrast" for t in tasks)
    if node and not cfg.get("targets"):
        raise ValueError("Config: knotenzentrierte Aufgaben ohne 'targets'.")
    if contrast and not _cfg(cfg, "reference_corpus", "ref_corpname"):
        raise ValueError("Config: korpuskontrastive Aufgaben ohne 'reference_corpus'.")
    if "key" in cfg or "api_key" in cfg:
        raise ValueError("Config darf keinen API-Key enthalten.")


# ============================================================================
# Output - XLSX-Workbooks (+ CSVs) mit run_config-Sheet
# ============================================================================

def _write_workbook(path: str, sheets: Dict[str, "pd.DataFrame"],
                    cfg: Dict[str, Any]) -> None:
    """Schreibt ein XLSX aus {Sheetname: DataFrame} + run_config-Sheet."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            (df if not df.empty else pd.DataFrame([{"note": "leer"}])).to_excel(
                writer, sheet_name=name[:31], index=False
            )
        flat = [{"parameter": k, "wert": json.dumps(v, ensure_ascii=False)
                 if isinstance(v, (list, dict)) else v}
                for k, v in cfg.items()]
        pd.DataFrame(flat).to_excel(writer, sheet_name="run_config", index=False)


def write_outputs(
    results: Dict[str, "pd.DataFrame"],
    cfg: Dict[str, Any],
    out_dir: Optional[str] = None,
    stamp: Optional[str] = None,
    announce: bool = True,
) -> str:
    """
    Schreibt die Ergebnisse. Layout via cfg['output_layout']:
      'by_task'   - ein Master-Workbook (Sheet je Aufgabe, alle Lexeme gestapelt)
                    + CSVs je Aufgabe. Tidy/analysefreundlich.
      'by_lexeme' - je Lexem ein Workbook (Sheet je Aufgabe, nur dieses Lemma)
                    + CSVs je Lexem/Aufgabe, in Unterordner by_lexeme/.
      'both'      - beides (Default).
    Dateinamen tragen Korpus-Kuerzel, Aufgaben-Klarnamen und Zeitstempel.
    Ein fester 'stamp' (von run() durchgereicht) sorgt dafuer, dass wiederholte
    Aufrufe (Zwischenspeicherung) dieselben Dateien ueberschreiben.
    """
    _ensure_deps()
    tag = _corpus_tag(_cfg(cfg, "corpus", "corpname", default="corpus"))
    stamp = stamp or _now_stamp()
    layout = _cfg(cfg, "output_layout", default="both")
    out_dir = out_dir or f"SE_{tag}_{stamp}"
    os.makedirs(out_dir, exist_ok=True)
    save_config(cfg, out_dir, filename=f"run_config_{tag}_{stamp}.json")
    # Zusaetzlich ein Template OHNE Lemmata fuer 'gleiche Parameter, andere Woerter'
    if cfg.get("targets"):
        save_config_template(cfg, out_dir, filename=f"run_config_template_{tag}_{stamp}.json")

    # --- Master (by_task): tidy, alle Lexeme je Aufgabe gestapelt ---
    # Master-CSV hat kein einzelnes Lemma -> Art__Korpus_Zeitstempel.
    if layout in ("by_task", "both"):
        _write_workbook(
            os.path.join(out_dir, f"SE_{tag}_{stamp}.xlsx"),
            {TASK_SHEET.get(t, f"task_{t}"): df for t, df in results.items()},
            cfg,
        )
        for t, df in results.items():
            fname = f"{TASK_TAG.get(t, t)}__{tag}_{stamp}.csv"
            df.to_csv(os.path.join(out_dir, fname), index=False)

    # --- Pro Lexem: je Lemma ein Workbook (Sheet je Aufgabe) + CSVs ---
    # Konvention: Lemma__Art__Korpus_Zeitstempel (Workbook: Lemma__Korpus, alle Arten).
    if layout in ("by_lexeme", "both"):
        lex_dir = os.path.join(out_dir, "by_lexeme")
        os.makedirs(lex_dir, exist_ok=True)
        lemmas = set()
        for df in results.values():
            if "lemma" in df.columns:
                lemmas.update(x for x in df["lemma"].dropna().unique())
        for lemma in sorted(lemmas):
            lp = _file_part(str(lemma))
            sheets: Dict[str, "pd.DataFrame"] = {}
            for t, df in results.items():
                if "lemma" not in df.columns:
                    continue
                sub = df[df["lemma"] == lemma].reset_index(drop=True)
                if sub.empty:
                    continue
                sheets[TASK_SHEET.get(t, f"task_{t}")] = sub
                csv = f"{lp}__{TASK_TAG.get(t, t)}__{tag}_{stamp}.csv"
                sub.to_csv(os.path.join(lex_dir, csv), index=False)
            if sheets:
                _write_workbook(
                    os.path.join(lex_dir, f"{lp}__{tag}_{stamp}.xlsx"), sheets, cfg)

    if announce:
        print(f"Ausgabe geschrieben: {out_dir}/  (Layout: {layout})")
    return out_dir


# ============================================================================
# Orchestrator
# ============================================================================

def run(cfg: Dict[str, Any], client: SEClient,
        out_dir: Optional[str] = None, write: bool = True) -> Tuple[Dict[str, "pd.DataFrame"], Optional[str]]:
    """
    Fuehrt die in cfg gewaehlten Aufgaben aus (c, d, Z, e, f) und gibt
    (results, out_dir) zurueck.

    INKREMENTELLES SPEICHERN: Bei write=True wird der Ausgabeordner zu Beginn
    angelegt und nach JEDER Aufgabe sowie nach JEDEM e/f-Zielwort der bisherige
    Stand auf Platte geschrieben (dieselben Dateien, fester Zeitstempel). Bei
    Abbruch (Kompilierungswartezeit, Verbindung, Kernel-Neustart) bleibt erhalten,
    was bis dahin gerechnet wurde. write=False rechnet nur, ohne zu schreiben.
    """
    _ensure_deps()
    results: Dict[str, "pd.DataFrame"] = {}
    corpname = _cfg(cfg, "corpus", "corpname")
    usesubcorp = _cfg(cfg, "focus_subcorpus", "usesubcorp")
    targets = cfg.get("targets", [])
    tasks = cfg["tasks"]

    tag = _corpus_tag(_cfg(cfg, "corpus", "corpname", default="corpus"))
    stamp = _now_stamp()
    if write:
        out_dir = out_dir or f"SE_{tag}_{stamp}"
        os.makedirs(out_dir, exist_ok=True)

    def flush():
        if write and results:
            write_outputs(results, cfg, out_dir=out_dir, stamp=stamp, announce=False)

    if "c" in tasks:
        results["c"] = task_c_word_sketch(
            client, corpname, targets, usesubcorp=usesubcorp,
            maxitems=_cfg(cfg, "ws_max_items_per_relation", "wsketch_maxitems",
                          default=DEFAULTS["wsketch_maxitems"]),
            clustercolls=_cfg(cfg, "ws_cluster_collocates", "wsketch_clustercolls",
                              default=DEFAULTS["clustercolls"]),
        )
        flush()

    if "d" in tasks:
        win = _cfg(cfg, "coll_window", "d_window", default=list(DEFAULTS["coll_window"]))
        results["d"] = task_d_window_collocation(
            client, corpname, targets, usesubcorp=usesubcorp,
            cfromw=win[0], ctow=win[1],
            cminfreq=_cfg(cfg, "coll_min_corpus_freq", "d_cminfreq", default=5),
            cminbgr=_cfg(cfg, "coll_min_cooccurrence_freq", "d_cminbgr", default=5),
            cmaxitems=_cfg(cfg, "coll_max_collocates", "n_collocates",
                           default=DEFAULTS["n_collocates"]),
            match_attr=_cfg(cfg, "node_match_attr", default="smart"),
        )
        flush()

    if "Z" in tasks:
        lemmas = cfg.get("compound_lemmas") or [t["lemma"] for t in targets]
        ci = _cfg(cfg, "comp_case_insensitive", default=None)
        wlicase = (0 if ci else 1) if ci is not None else _cfg(
            cfg, "wlicase", default=DEFAULTS["wlicase"])
        results["Z"] = task_Z_compounds(
            client, corpname, lemmas, usesubcorp=usesubcorp,
            wlminfreq=_cfg(cfg, "comp_min_freq", "wlminfreq", default=5),
            wlmaxitems=_cfg(cfg, "comp_max_items", default=DEFAULTS["wl_maxitems"]),
            wlicase=wlicase,
        )
        flush()

    if "e" in tasks:
        def _partial_e(df):
            results["e"] = df
            flush()
        results["e"] = run_keywords(client, cfg, usengrams=0, on_partial=_partial_e)
        flush()

    if "f" in tasks:
        def _partial_f(df):
            results["f"] = df
            flush()
        results["f"] = run_keywords(client, cfg, usengrams=1, on_partial=_partial_f)
        flush()

    if write:
        print(f"Fertig. Ausgabeordner: {out_dir}")
    return results, out_dir


def prepare_subcorpora(cfg: Dict[str, Any], client: SEClient) -> List[Dict[str, Any]]:
    """
    GETRENNTE VORBEREITUNG: legt nur die Diskurs-Subkorpora an (keine
    Keyword-Berechnung). Einmal fuer viele Lemmata laufen lassen; die Subkorpora
    kompilieren danach serverseitig und stehen beim spaeteren e/f-Lauf (mit
    denselben Parametern) bereits abfragebereit zur Wiederverwendung.

    Loescht nichts und ueberspringt nichts - uebergrosse Subkorpora werden nur
    informativ markiert (Anteil am Korpus). Gibt eine Zusammenfassung je Subkorpus
    zurueck: {lemma, subcorpus, size, share, over_threshold}.
    """
    if not cfg.get("discourse_mode"):
        client._log("[prepare] discourse_mode ist aus - es gibt keine Subkorpora anzulegen.")
        return []
    corpname = _cfg(cfg, "corpus", "corpname")
    targets = cfg.get("targets", []) or []
    struct = _cfg(cfg, "discourse_subcorpus_struct", "discourse_struct", default="doc")
    per_word = bool(_cfg(cfg, "discourse_subcorpus_per_word", "discourse_per_word", default=True))
    match_attr = _cfg(cfg, "node_match_attr", default="smart")
    corpus_size = _to_int(_cfg(cfg, "corpus_size_tokens", default=None))
    max_share = float(_cfg(cfg, "kw_max_subcorpus_share",
                           default=DEFAULTS["kw_max_subcorpus_share"]))
    sm = SubcorpusManager(client, corpname)
    summary: List[Dict[str, Any]] = []

    def record(label: str, name: str, size: Optional[int]) -> None:
        share = (size / corpus_size) if (corpus_size and size) else None
        over = bool(share is not None and share > max_share)
        info = ""
        if share is not None:
            info = f" = {share:.2%} des Korpus" + (
                f" [UEBER Schwelle {max_share:.0%} -> e/f wird es ueberspringen]" if over else "")
        client._log(f"[prepare] '{label}' -> '{name}': {size} Token{info}")
        summary.append({"lemma": label, "subcorpus": name, "size": size,
                        "share": share, "over_threshold": over})

    if per_word:
        for t in targets:
            lemma, lpos = t["lemma"], t.get("lpos") or None
            name = sm.make_name(lemma)
            q = sm.cql_for_lemma(lemma, lpos, match_attr=match_attr)
            client._log(f"[prepare] '{lemma}': Subkorpus '{name}' (struct={struct}) ...")
            size = sm.create_query_subcorpus(name, q, struct=struct, wait=True)
            record(lemma, name, size)
    else:
        lemmas = [t["lemma"] for t in targets]
        name = sm.make_name("union_" + "_".join(_sanitize(l) for l in lemmas))[:60]
        cql = "|".join(sm.cql_for_lemma(l, match_attr=match_attr) for l in lemmas)
        client._log(f"[prepare] Union {lemmas}: Subkorpus '{name}' (struct={struct}) ...")
        size = sm.create_query_subcorpus(name, cql, struct=struct, wait=True)
        record("(union)", name, size)

    n_over = sum(1 for s in summary if s["over_threshold"])
    client._log(f"[prepare] fertig: {len(summary)} Subkorpus(e) angelegt/wiederverwendet"
                + (f", davon {n_over} ueber der Schwelle." if n_over else "."))
    return summary


def main() -> None:
    """Interaktiver Einstieg (Colab/Terminal). run() schreibt inkrementell."""
    cfg, client = build_config_interactive()
    results, out_dir = run(cfg, client)


if __name__ == "__main__":
    main()


Writing sketch_engine_wrapper.py


In [ ]:
# (2) Engine importieren + Client aufbauen (Auth frueh = fail-fast)
import importlib, sketch_engine_wrapper as se
importlib.reload(se)
user, key = se.resolve_credentials()   # Colab-Secret -> Env -> getpass
client = se.SEClient(user, key, verbose=True)
print("Auth gesetzt fuer Benutzer:", user)

Sketch-Engine-Benutzername: LS_Fr_It_IUED
Sketch-Engine-API-Key (verdeckt): ··········
Auth gesetzt fuer Benutzer: LS_Fr_It_IUED


## Smoke-Test

corp_info/gramrels/wsketch/wordlist/collx sind bestaetigt. **Neu:**
extract_keywords + subcorp (Aufgabe e).

In [ ]:
# (3) Test-Parameter
TEST_CORPUS = "user/LS_Fr_It_IUED/all_v_d_k"   # kleines Korpus
BIG_CORPUS  = "preloaded/detenten23_rft3_1"    # grosses Korpus
REF_CORPUS  = BIG_CORPUS   # Referenz fuer e; MUSS gleiche Verarbeitung haben wie Fokus!
TEST_LEMMA  = None         # wird unten automatisch gewaehlt

In [ ]:
# Verify 1+2 - corp_info-Groesse & gramrels=1-Namen
info = client.corp_info(TEST_CORPUS)
print("info['sizes']:", info.get("sizes"))
print("-> Groesse:", se._format_tokens(client.corpus_size(TEST_CORPUS)))
names = client.gramrel_names(TEST_CORPUS)
print("Relationsnamen:", len(names), "| Beispiele:", names[:5])

In [ ]:
# Verify 3+4 - Inhaltslemma finden, wsketch + wordlist
wl = client.wordlist(TEST_CORPUS, wlattr="lemma", wlpat=".*", wlminfreq=5, wlmaxitems=50)
wit = se._first_present(wl, ["Items","items"], []) or []
print("wordlist Item-Keys:", list(wit[0].keys()) if wit else "(leer)")
cands = [se._first_present(it, ["str","word"]) for it in wit]
cands = [c for c in cands if c and c.isalpha() and len(c) > 3]
for cand in cands:
    dws = client.wsketch(TEST_CORPUS, cand, maxitems=10)
    for rel in (dws.get("Gramrels") or []):
        w = rel.get("Words") or []
        if w and not str(rel.get("name","")).lower().startswith("construction"):
            TEST_LEMMA = cand
            print("-> Inhaltslemma:", repr(cand), "| ws_status:", dws.get("ws_status"),
                  "| Kollokat-Keys:", list(w[0].keys()))
            break
    if TEST_LEMMA: break
print("TEST_LEMMA =", repr(TEST_LEMMA))

In [ ]:
# Verify d - collx-Struktur (bestaetigt: str, Stats mit n=d/m/t, freq/coll_freq)
qd = "q" + se.SubcorpusManager.cql_for_lemma(TEST_LEMMA or "x")
cx = client.collx(TEST_CORPUS, qd, cfromw=-5, ctow=5, cminfreq=3, cminbgr=3,
                  cbgrfns=["d","m","t"], cmaxitems=5)
cit = se._first_present(cx, ["Items","items"], []) or []
print("collx Item[0]:", cit[0] if cit else "(leer)")

In [ ]:
# Verify e - extract_keywords-Struktur (PLAIN-Modus, KEIN Schreibvorgang)
# REF_CORPUS muss dieselbe Verarbeitung haben wie TEST_CORPUS, sonst lehnt SE ab.
try:
    kw = client.extract_keywords(TEST_CORPUS, REF_CORPUS, simple_n="5",
                                 minfreq=5, max_keywords=10)
    print("extract_keywords Top-Level-Keys:", list(kw.keys())[:20])
    its = se._first_present(kw, ["Items","keywords","data","items"], []) or []
    print("Anzahl Items:", len(its))
    if its:
        print("Item-Keys:", list(its[0].keys()))
        print("Item[0]:", its[0])
    print("\n-> simple_n=5 akzeptiert? Schluessel fuer Keyword/Keyness/Freq?")
except se.SketchEngineError as e:
    print("Fehler:", e)
    print("-> ggf. REF_CORPUS auf kompatibles Korpus aendern, oder simple_n=1 testen.")

In [ ]:
# Verify e (Diskurs+subcorp) - OPTIONAL: legt ein PERSISTENTES Subkorpus an (auto_disc_*)
RUN_DISCOURSE_SMOKE = False
if RUN_DISCOURSE_SMOKE and TEST_LEMMA:
    sm = se.SubcorpusManager(client, TEST_CORPUS)
    name = sm.make_name(TEST_LEMMA)
    sm.create_query_subcorpus(name, sm.cql_for_lemma(TEST_LEMMA), struct="doc", wait=True)
    print("Subkorpora jetzt:", client.list_subcorpora(TEST_CORPUS))
    kw = client.extract_keywords(TEST_CORPUS, REF_CORPUS, usesubcorp=name,
                                 simple_n="5", minfreq=5, max_keywords=10,
                                 wlblacklist=TEST_LEMMA)
    its = se._first_present(kw, ["Items","keywords","data","items"], []) or []
    print("Diskurs-Keywords:", [se._first_present(i,["str","word"]) for i in its[:8]])
else:
    print("Diskurs-Smoke uebersprungen (RUN_DISCOURSE_SMOKE=False).")

## (Optional) Getrennte Vorbereitung: nur Subkorpora anlegen

Entkoppelt den teuren Schritt. Diese Zelle legt **nur** die Diskurs-Subkorpora an (keine Keyword-Berechnung) - einmal fuer eine Reihe von Lemmata laufen lassen; sie kompilieren danach serverseitig. Der spaetere `e`/`f`-Lauf weiter unten verwendet sie - **mit denselben Parametern** (am besten dasselbe Template + dieselben Lemmata) - bereits abfragebereit wieder. Setzt `discourse_mode=on` voraus. Ueberspringen, wenn nicht gebraucht.

In [ ]:
# (3b, optional) NUR Subkorpora vorbereiten - keine Keywords
prep_cfg, prep_client = se.build_config_interactive()
prep_summary = se.prepare_subcorpora(prep_cfg, prep_client)
for s in prep_summary:
    share = f"{s['share']:.2%}" if s.get("share") is not None else "?"
    mark = "  <-- ueber Schwelle (e/f ueberspringt es)" if s.get("over_threshold") else ""
    print(f"  {s['lemma']:28} {s['subcorpus']:40} {s['size']!s:>14} Token  {share}{mark}")

## Produktiver Lauf

Interaktive Config -> Lauf -> Ausgabe. Ausgefuehrt: **c**, **d**, **e**, **f**, **Z** (vollstaendig). Ausgabe-Layout waehlbar (Master-Tabelle und/oder pro Lexem).

In [ ]:
# (4) Interaktive Config -> Lauf -> Ausgabe (run() speichert inkrementell)
cfg, client = se.build_config_interactive()
results, out_dir = se.run(cfg, client)   # schreibt nach jeder Aufgabe / jedem e,f-Lemma
print("Fertig. Ausgabeordner:", out_dir)

Bestehende Config-JSON laden? (Pfad oder leer fuer interaktiv) []: /content/Phys_TP.json
Sketch-Engine-Benutzername: LS_Fr_It_IUED
Sketch-Engine-API-Key (verdeckt): ··········
  Template erkannt (Config ohne Zielwoerter) - bitte Lemmata eingeben.
Zielwoerter (eine Zeile je Eintrag); leere Zeile beendet.
  Einzelwort optional mit POS am Ende: 'Tisch -n'
  Mehrworteinheit einfach als Phrase: 'Dominikanische Republik'
  (Mehrwort wirkt nur fuer d/e/f; c und Z ueberspringen Phrasen.)
   > Masse
   > Gewicht
   > Schwere -n
   > Trägheit
   > schwer
   > 
  Geladen: Korpus=user/LS_Fr_It_IUED/all_v_d_k | Aufgaben=['c', 'd', 'Z'] | Zielwoerter=['Masse', 'Gewicht', 'Schwere', 'Trägheit', 'schwer']
Etwas an der geladenen Config aendern (sonst 1:1 uebernehmen)? [j/N]: N

=== Konfiguration (ohne API-Key) ===
{
  "created": "2026-05-23T00:11:59",
  "username": "LS_Fr_It_IUED",
  "tasks": [
    "c",
    "d",
    "Z"
  ],
  "corpus": "user/LS_Fr_It_IUED/all_v_d_k",
  "corpus_size_tokens": 1291744,
 

In [ ]:
# (5) Ergebnisse herunterladen (Colab)
import shutil, os
try:
    from google.colab import files  # type: ignore
    shutil.make_archive(out_dir.rstrip("/"), "zip", out_dir)
    files.download(out_dir.rstrip("/") + ".zip")
except Exception as e:
    print("Kein Colab-Download (lokal?). Ordner:", os.path.abspath(out_dir))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>